# 06_llm_prompt_design 

LLM prompt iteration on 20-post dev probe before full eval run. Keep cost under .

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np

root_dir = Path(os.getcwd()).parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
# Eigen scripts

from src.data import load_splits
from src.llm.prompts import build_prompt
from src.llm.openai_client import classify as classify_openai
from src.llm.openai_client import classify as classify_gpt
from src.llm.gemini_client import classify as classify_gemini

# Laad de API keys
load_dotenv()
print("API Keys geladen:", "GOOGLE_API_KEY" in os.environ)

API Keys geladen: True


In [2]:
train_df, val_df, test_df = load_splits()

# Test op 20 posts
test_subset = train_df.sample(20, random_state=42)

# Haal de few-shot voorbeelden uit de rest van de training set
few_shot_examples = (
    train_df.drop(test_subset.index)
    .groupby("label_id")
    .apply(lambda g: g.sample(2, random_state=42))
    .reset_index(drop=True)[["text", "label"]]
    .to_dict("records")
)

print(f"Subset van {len(test_subset)} posts klaar voor test.")

Subset van 20 posts klaar voor test.


In [3]:
# Check de prompt voor de eerste post uit je subset
sample_post = test_subset.iloc[0]['text']
prompt_check = build_prompt(variant="few_shot", post=sample_post, few_shot_examples=few_shot_examples)

print("--- VOORBEELD PROMPT ---")
print(prompt_check)

--- VOORBEELD PROMPT ---
You are an assistant that classifies social media posts based on the severity of depression symptoms.
Classify the following post into one of four categories based on the described symptoms:
Choose one label that best fits the symptoms described in the post:
- minimum: no or minimal symptoms
- mild: mild symptoms, still functions mostly normally
- moderate: clear symptoms, noticeably reduced functioning
- severe: severe symptoms, significant suffering or dysfunction

Post: "I've been frustrated in the past by 2AM pulling it down and running too soon. But because I want to have my cake and eat it too, if on that 3rd down play he cut left, he's walking for a 1st down."
Label: "minimum"

Post: "But now, I can't. I literally have ONE therapist I can see. And ONE psychiatrist (who is actually a nurse practitioner). I have completely given up on getting the correct mental health treatment. I am doing the best I can."
Label: "minimum"

Post: "i rlly am tired of starti

CHATGPT TEST

In [4]:

results = []

for i, (idx, row) in enumerate(test_subset.iterrows()):
    #Bouw de prompt (Chain of Thought variant zoals gevraagd)
    prompt = build_prompt(variant="chain_of_thought", post=row["text"], few_shot_examples=few_shot_examples)
    # Roep de OpenAI client aan met de config voor GPT-5
    res = classify_gpt(prompt, config={"model": "gpt-5.4-mini", "variant": "chain_of_thought", "max_tokens": 1000}
    )
    
    res['true_label'] = row['label']
    res['post_id'] = idx
    
    results.append(res)
    
    print(f"Post {i+1}: GPT-5.4-mini koos '{res['label']}' (Echt: {res['true_label']})")

# Maak de uiteindelijke DataFrame
test_results_df = pd.DataFrame(results)

Post 1: GPT-5.4-mini koos 'mild' (Echt: mild)
Post 2: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 3: GPT-5.4-mini koos 'moderate' (Echt: severe)
Post 4: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 5: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 6: GPT-5.4-mini koos 'mild' (Echt: mild)
Post 7: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 8: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 9: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 10: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 11: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 12: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 13: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 14: GPT-5.4-mini koos 'mild' (Echt: mild)
Post 15: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 16: GPT-5.4-mini koos 'severe' (Echt: moderate)
Post 17: GPT-5.4-mini koos 'minimum' (Echt: minimum)
Post 18: GPT-5.4-mini koos 'mild' (Echt: mild)
Post 19: GPT-5.4-mini koos 'mild' (Echt: mild)
Post 20: GPT-5.4-mini koos 'mini

GEMINI TEST

In [ ]:
results = []

for i, (idx, row) in enumerate(test_subset.iterrows()):
    # We testen Chain of Thought omdat die het meest complex is
    prompt = build_prompt(variant="chain_of_thought", post=row["text"], few_shot_examples=few_shot_examples)
    
    # Roep de client aan
    res = classify_gemini(prompt, config={"model": "gemini-3-flash-preview", "variant": "chain_of_thought", "temperature": 0.0, "max_tokens": 1000})
    
    # Sla resultaat op
    res['true_label'] = row['label']
    results.append(res)
    print(res['label'])
    # print(f"Post {i+1}/20: Model koos '{res['label']}' (Echt: {res['true_label']})")

test_results_df = pd.DataFrame(results)

unknown
minimum
unknown
minimum
minimum
unknown
unknown
minimum
unknown
unknown
unknown
minimum
minimum


In [ ]:
avg_cost = test_results_df['cost_usd'].mean()
total_estimated = avg_cost * 1040 * 3 * 2 # 1040 posts, 3 varianten, 2 modellen

print(f"Gemiddelde kosten per call: ${avg_cost:.4f}")
print(f"Geschatte totale kosten: ${total_estimated:.2f}")

# Laat de eerste 5 resultaten zien inclusief de reasoning
test_results_df[['true_label', 'label', 'reasoning', 'cost_usd']].head()